> 📝 **Planning note (author use -- remove before publishing)**
>
> **Intended contents:** auto-metadata, compliance checks (IOOS + custom), manual/custom metadata addition, dtype/fill-value finalization for NetCDF export.

**To do:** ~~add the missing `ctd` import~~ (checked -- `ctd` is never used here; trimmed the unused imports instead); ~~fix three separate SyntaxErrors from prose sitting in code cells~~ (done -- converted to markdown); ~~fix the `ds.CNDC['long_name']` line~~ (done); ~~remove the duplicated `nans_to_fill_value` call~~ (done); ~~typo cleanup~~ (done). Remaining: narrative throughout, and the placeholder bullets near the end.

<img src="../../data/images/hiaoos_learning_moored.png" width="300" align="right">


# Preparing data for publication

The state of the art for publishing ocean data is the NetCDF format with community standard metadata conventions (CF and ACDD).

In [1]:
from kval.data import moored
from kval.metadata import conventionalize

> ✏️ **Review note** — this now points at `AT200_21_22_SBE37_15252_113m_edited.nc`, **which does not exist yet** -- notebook 4 only processes the RBR/AT800 record. See the note at the end of notebook 4. Until that file is produced, this cell will fail.

Why it matters here and not just tidiness: the `_from_raw` AT200 file still contains on-deck measurements at both ends (`PRES` near $-$0.4 dbar, `TEMP` near 10 °C at the end of the record). Any statistic computed over it is partly a statistic about air.

In [4]:
ds = moored.load_nc('../../data/moored_CTD_test_data/intermediate_data/AT200_21_22_SBE37_20773_49m_edited.nc')

## Add some metadata

In [5]:
ds = moored.metadata_auto(ds)

## Run quick checks

In [6]:
moored.compliance_checks_ioos(ds)

In [7]:
moored.compliance_checks_custom(ds)


----------------------------------
⚠️ Dataset has 5 issue(s)
----------------------------------

❌❌❌ MISSING REQUIRED GLOBAL ATTRIBUTES ❌❌❌
title, summary, creator_name, creator_email, institution
⮕ These MUST be added for CF/ACDD compliance

⚠️ 64-bit types (32-bit is recommended):
PRES, CNDC, TEMP, PSAL, TIME
   ⮕ Suggestion: use kval.conventionalize.convert_64_to_32(ds)

⚠️ Suspicious/missing _FillValue (including NaNs, which are discouraged):
PRES, CNDC, TEMP, PSAL, TIME, LATITUDE, LONGITUDE
   ⮕ Suggestion: use kval.conventionalize.nans_to_fill_value(ds)

❌ Missing 'processing_level':
PRES, CNDC, TEMP, PSAL
   ⮕ Suggestion: add globally or on all relevant variables

⚠️ Missing 'QC_indicator' (not strictly required):
PRES, CNDC, TEMP, PSAL

⚠️ Missing recommended global attributes:
product_version, id, license, project, doi, acknowledgment, references, data_set_progress, cruise_name, ship, area, location, data_assembly_center, creator_type, creator_url, creator_institution, publis

## Add some custom metadata

#### In xarray

There are a few ways to add attributes. You can set them directly in `xarray`:

In [10]:
ds.attrs['title'] = 'Mock title' # Set a global attribute
ds.CNDC.attrs['long_name'] = 'Sea water electrical conductivity' # Set a variable-level attribute

#### Using kval helper functions

... or use some helper functions if you prefer. These prompt you for the value:

In [11]:
ds = conventionalize.set_glob_attr(ds, 'title')

In [12]:
ds = conventionalize.set_var_attr(ds, 'TEMP', 'standard_name')

#### Using input file


### Finalize dataset

In [ ]:
ds = moored.add_now_as_date_created(ds)

In [ ]:
ds = moored.convert_64_to_32(ds)

In [ ]:
ds = moored.nans_to_fill_value(ds)

- floats, fillvals, etc
- sort and stuff

In [ ]:
moored.compliance_checks_custom(ds)

### Export the final file

> ✏️ **Review note** — the notebook stopped at the compliance check and never wrote anything out, which seemed like the one thing a "preparing data for publication" notebook has to end with. Added the export below -- adjust the path and filename to whatever you want the published product to be called.

In [ ]:
out_path = '../../data/moored_CTD_test_data/'
out_name = 'AT200_21_22_SBE37_15252_113m.nc'

moored.to_netcdf(ds, out_path, out_name)